# Practical 6 — k-Nearest Neighbours and Classifier Evaluation

**Lectures:** 12. Binary Classification & Instance-based Learning · 13. Classifier Evaluation
(with k-fold cross-validation from 20. Cross Validation and Hyperparameter Tuning)

k-NN is the classic **instance-based** ("lazy") learner: training just stores the data; all the work
happens at prediction time. In this practical you will

1. implement k-NN from scratch and check it against scikit-learn;
2. implement the confusion matrix, accuracy, precision, recall, specificity, F1, the ROC curve and AUC yourself;
3. implement **k-fold cross-validation** to choose $k$.

We use the **Wisconsin breast cancer** dataset: 569 tumours, 30 features computed from cell images,
label *malignant* or *benign*. We treat **malignant as the positive class (1)**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X = data.data
y = (data.target == 0).astype(int)          # 1 = malignant (positive), 0 = benign
print(X.shape, "positives:", y.sum(), "negatives:", (1 - y).sum())

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

## 1. k-NN from scratch (≈30 min)
To classify $\mathbf x$: compute its distance to every training example, take the $k$ closest, and
predict the majority label among them. For a probability-like **score**, use the fraction of the $k$
neighbours that are positive.

**TODO 1.1** Implement `pairwise_distances(A, B)` returning the matrix of Euclidean distances between
every row of `A` and every row of `B`, **without Python loops**. Hint:
$\lVert\mathbf a-\mathbf b\rVert^2 = \lVert\mathbf a\rVert^2 + \lVert\mathbf b\rVert^2 - 2\,\mathbf a^\top\mathbf b$.

**TODO 1.2** Complete `KNNClassifier` (binary labels 0/1). `np.argsort` or `np.argpartition` will help.

In [ ]:
def pairwise_distances(A, B):
    # YOUR CODE HERE
    raise NotImplementedError

class KNNClassifier:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        # YOUR CODE HERE
        return self

    def predict_proba(self, X):
        """Fraction of the k nearest neighbours that belong to class 1."""
        # YOUR CODE HERE
        raise NotImplementedError

    def predict(self, X):
        return (self.predict_proba(X) > 0.5).astype(int)

**TODO 1.3** Compare your classifier with `KNeighborsClassifier(n_neighbors=5)` — on **raw** features and on
**standardised** features (fit the scaler on the training set). Do the predictions agree? How much does
scaling matter for k-NN and why?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

sc = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = sc.transform(X_tr), sc.transform(X_te)
# YOUR CODE HERE

## 2. Evaluating a classifier (≈35 min)
**TODO 2.1** Implement `confusion(y_true, y_pred)` returning `(TP, FP, TN, FN)`, and use it to compute
accuracy, precision (PPV), recall (sensitivity / TPR), specificity (TNR), negative predictive value and
$F_1 = 2\cdot\frac{\text{precision}\cdot\text{recall}}{\text{precision}+\text{recall}}$.
Check against `sklearn.metrics` (`confusion_matrix`, `precision_score`, `recall_score`, `f1_score`).

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

def confusion(y_true, y_pred):
    # YOUR CODE HERE
    raise NotImplementedError

def metrics(y_true, y_pred):
    TP, FP, TN, FN = confusion(y_true, y_pred)
    # YOUR CODE HERE
    raise NotImplementedError

knn = KNNClassifier(5).fit(X_tr_s, y_tr)
y_pred = knn.predict(X_te_s)
print("TP, FP, TN, FN =", confusion(y_te, y_pred))
print({k: round(float(v), 4) for k, v in metrics(y_te, y_pred).items()})
# YOUR CODE HERE

**TODO 2.2** In this medical setting, which is worse: a false positive or a false negative? Which metric
would you therefore watch most closely?

**TODO 2.3 — The accuracy paradox.** A "classifier" that always predicts *benign* — what are its accuracy,
recall and precision on the test set?

In [ ]:
# YOUR CODE HERE

**TODO 2.4 — ROC and AUC.** Changing the decision threshold $t$ on the score trades recall against
specificity. Write `roc_curve_manual(y_true, scores)` that, for each distinct threshold, returns the
false-positive rate $\text{FPR}=1-\text{specificity}$ and true-positive rate (recall), and compute the
area under the curve with the trapezium rule (`np.trapezoid`). Plot your ROC curve for $k=5$ and
$k=15$ and compare with `roc_auc_score`.

In [ ]:
from sklearn.metrics import roc_auc_score

def roc_curve_manual(y_true, scores):
    # YOUR CODE HERE
    raise NotImplementedError

# YOUR CODE HERE

## 3. Choosing k with k-fold cross-validation (≈30 min)
We must **not** choose $k$ by looking at the test set. Instead, split the *training* set into $K$ folds;
train on $K-1$ of them, validate on the remaining one, rotate, and average.

**TODO 3.1** Implement `kfold_indices(n, K, seed)` (shuffle once, then cut into $K$ nearly-equal folds) and
`cross_val_accuracy(model_factory, X, y, K)`. **Important:** the scaler must be fitted *inside* each fold on
the $K-1$ training folds only. Use 10-fold CV to evaluate $k = 1, 3, 5, \dots, 31$ and plot mean CV accuracy
(with ± one standard deviation) against $k$. Then compare with `cross_val_score` on a
`make_pipeline(StandardScaler(), KNeighborsClassifier(k))`.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

def kfold_indices(n, K, seed=0):
    # YOUR CODE HERE
    raise NotImplementedError

def cross_val_accuracy(k, X, y, K=10, seed=0):
    scores = []
    folds = kfold_indices(len(y), K, seed)
    for i in range(K):
        # YOUR CODE HERE
        raise NotImplementedError
    return np.array(scores)

ks = list(range(1, 32, 2))
# YOUR CODE HERE

## 4. Extension: multi-class k-NN
k-NN handles several classes naturally (majority vote). Generalise your classifier to any labels
(hint: `np.unique(..., return_counts=True)` or `scipy.stats.mode`), apply it to the **iris** dataset
(`sklearn.datasets.load_iris`), and show the 3×3 confusion matrix. Which two species get confused?

In [ ]:
# YOUR CODE HERE

## Lab report checklist (Practical 6)
1. Evidence that your k-NN matches scikit-learn, and the accuracy with and without scaling (explain the difference).
2. Your confusion matrix and the six metrics for $k=5$; which metric matters most for this problem and why; the accuracy paradox result.
3. ROC curves and AUC for two values of $k$.
4. The cross-validation plot, your chosen $k$, and the final test-set metrics. Explain why $k$ must be chosen by cross-validation rather than on the test set.